# Tensors

Tensors are a specialized data structure that are very similar to arrays and matrices. In PyTorch, we use tensors to encode the inputs and outputs of a model, as well as the model’s parameters.

Tensors are similar to NumPy’s ndarrays, except that tensors can run on GPUs or other hardware accelerators. In fact, tensors and NumPy arrays can often share the same underlying memory, eliminating the need to copy data (see Bridge with NumPy). Tensors are also optimized for automatic differentiation (we’ll see more about that later in the Autograd section). If you’re familiar with ndarrays, you’ll be right at home with the Tensor API. If not, follow along!

In [1]:
import torch
import numpy as np

## Initialize the Tensor

Tensors can be initialized in various ways. Take a look at the following examples:

In [2]:
# Directly from data
data = [[1, 2], [3, 4]]
data_tensor = torch.tensor(data)
print(data_tensor)

tensor([[1, 2],
        [3, 4]])


In [3]:
# From a NumPy ndarray
np_array = np.array(data)
np_tensor = torch.tensor(np_array)
print(np_tensor)

tensor([[1, 2],
        [3, 4]])


In [4]:
# From another tensor
ones_tensor = torch.ones_like(data_tensor)
print(ones_tensor)

tensor([[1, 1],
        [1, 1]])


In [6]:
# rand_like: [0,1) uniform distribution
rand_tensor = torch.rand_like(data_tensor, dtype=torch.float32) 
print(rand_tensor)

# randn_like:(-inf, +inf) standard gaussian distribution
randn_tensor = torch.randn_like(data_tensor, dtype=torch.float32)
print(randn_tensor)

tensor([[0.5308, 0.4888],
        [0.5214, 0.0386]])
tensor([[-0.5627, -0.1110],
        [-0.0603,  0.6903]])


In [7]:
# With random or constant values
shape = (3, 2)

zeros = torch.zeros(shape)
ones = torch.ones(shape)
rand = torch.rand(shape)

print(f"{zeros}\n{ones}\n{rand}\n")

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])
tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])
tensor([[0.7024, 0.9594],
        [0.2002, 0.6366],
        [0.5779, 0.8406]])



## Attributes of a Tensor

Tensor attributes describe their shape, datatype, and the device on which they are stored.

In [8]:
tensor = torch.rand(2, 4)

print("The shape of the tensor is: ", tensor.shape)
print("The datatype of the tensor is: ", tensor.dtype)
print("The device of the tensor is: ", tensor.device)

The shape of the tensor is:  torch.Size([2, 4])
The datatype of the tensor is:  torch.float32
The device of the tensor is:  cpu


In [ ]:
# differences between size and shape
"""
Both return the shape of the tensor as a torch.Size object.
.shape is an attribute, while .size() is a method.
"""
print("tensor.shape: ",tensor.shape)
print("tensor.size(): ", tensor.size())

tensor.shape:  torch.Size([2, 4])
tensor.size():  torch.Size([2, 4])


## Operations on Tensors

Over 1200 tensor operations, including arithmetic, linear algebra, matrix manipulation (transposing, indexing, slicing), sampling and more are comprehensively described here.

Each of these operations can be run on the CPU and Accelerator such as CUDA, MPS, MTIA, or XPU. If you’re using Colab, allocate an accelerator by going to Runtime > Change runtime type > GPU.

By default, tensors are created on the CPU. We need to explicitly move tensors to the accelerator using `.to` method (after checking for accelerator availability). Keep in mind that copying large tensors across devices can be expensive in terms of time and memory!

In [13]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
    tensor = tensor.to(device)
    print(f"Current accelerator is {device}")

Current accelerator is cuda


**Standard numpy-like indexing and slicing:**

In [16]:
tensor = torch.ones(4, 4)
print("original tensor: \n", tensor)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")

# change the second column value 
tensor[:,1] = 0
print("tensor after operation: \n",tensor)

original tensor: 
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor after operation: 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Joining tensors** 

You can use `torch.cat` to concatenate a sequence of tensors along a given dimension. See also `torch.stack`, another tensor joining operator that is subtly different from `torch.cat`.

In [18]:
tensor1 = torch.ones(2, 3)
tensor0 = torch.zeros(2, 3)

cat_tensor_dim0 = torch.cat([tensor1, tensor0], dim=0)
cat_tensor_dim1 = torch.cat([tensor1, tensor0], dim=1)
print(cat_tensor_dim0)
print(cat_tensor_dim1)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.]])


**Arithmetic operations**

In [19]:
tensor = torch.rand(2, 4)
print("tensor: \n", tensor)

tensor_transpose = tensor.T
print("transposed tensor: \n", tensor_transpose)

tensor: 
 tensor([[0.8985, 0.0250, 0.0184, 0.5545],
        [0.4092, 0.0814, 0.9724, 0.2833]])
transposed tensor: 
 tensor([[0.8985, 0.4092],
        [0.0250, 0.0814],
        [0.0184, 0.9724],
        [0.5545, 0.2833]])


In [27]:
tensor_one = torch.tensor([1., 1.])
tensor_zero = torch.tensor([0., 2.])

# @ computes the matrix multiplication 
result_at = tensor_one @ tensor_zero
result_matmul = tensor_one.matmul(tensor_zero)
result_operation_matmul = torch.rand_like(result_at, dtype=torch.float)
torch.matmul(tensor_one, tensor_zero, out=result_operation_matmul)

print("result of tensor1 @ tensor0 : \n", result_at, "\n", result_matmul, "\n", result_operation_matmul)

# * computes the element-wise product
result_star = tensor_one * tensor_zero
result_mul = tensor_one.mul(tensor_zero)
result_operation_mul = torch.rand_like(result_star, dtype=torch.float)
torch.mul(tensor_one, tensor_zero, out=result_operation_mul)
print("result of tensor0 * tensor1: \n", result_star, "\n", result_mul, "\n", result_operation_mul)


result of tensor1 @ tensor0 : 
 tensor(2.) 
 tensor(2.) 
 tensor(2.)
result of tensor0 * tensor1: 
 tensor([0., 2.]) 
 tensor([0., 2.]) 
 tensor([0., 2.])


**Single Element Tensors**

 If you have a one-element tensor, for example by aggregating all values of a tensor into one value, you can convert it to a Python numerical value using `item()`

In [29]:
tensor = torch.ones(4, 4)
add = tensor.sum()
result = add.item()
print(result, type(result))

16.0 <class 'float'>


**In-place operations** 

Operations that store the result into the operand are called in-place. They are denoted by a `_` suffix. For example: `x.copy_(y)`, `x.t_()`, will change `x`.

In [30]:
print(tensor)
tensor.add_(5)
print(tensor)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]])


## Bridge with NumPy

Tensors on the CPU and NumPy arrays can share their underlying memory locations, and changing one will change the other.

**Tensor to NumPy array**

In [31]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


A change in the tensor reflects in the NumPy array.

In [32]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


**Numpy array to Tensor**

In [34]:
n = np.ones(5)
t = torch.from_numpy(n)
print(f"t: {t}\nn: {n}")

t: tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
n: [1. 1. 1. 1. 1.]


Changes in the NumPy array reflects in the tensor.

In [35]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]
